# 3 — Model Training (Camber Cloud — Full Resources)
## FireSpreadNet · Next Day Wildfire Spread

**Hardware target**: Camber cloud platform (T4/V100/A100 GPU, 16+ GB VRAM)  
**Protocol**: Holdout (train/val/test) — no cross-validation  
**Objective**: Best possible results for a research publication

### Approach

With more VRAM and compute budget on Camber, we can:

| Aspect | Local GTX 1050 | Camber (this notebook) |
|--------|---------------|----------------------|
| Batch size | 4–8 | 16–32 |
| Training epochs | 40 | 80 |
| HP trials | 3 per model | 5 per model |
| PI-CCA batch | 4 (accum=2) | 16 native |
| Test-time augmentation | No | Yes (8-fold TTA) |
| MC-Dropout uncertainty | No | Yes (PI-CCA) |
| Models | 3 DL + CA | 3 DL + CA + ablation |

### Models compared

| Model | Type | Params | Notes |
|-------|------|--------|-------|
| CA | Physics baseline | 0 | Rothermel rules, no training |
| ConvLSTM | Deep Learning | ~350 K | Spatio-temporal baseline |
| U-Net + Attention | Deep Learning | ~2.1 M | Dense segmentation |
| PI-CCA | Hybrid | ~1.5 M | Physics-informed (main contribution) |

### References

1. Huot et al. (2022). *Next Day Wildfire Spread*. IEEE TGRS.  
2. Luo et al. (2026). *U-Net + Hadamard Transform for Wildfire Prediction*. arXiv:2602.11672.  
3. Anastasiou et al. (2025). *Wildfire spread forecasting with Deep Learning*. arXiv:2505.17556.  
4. Zhou et al. (2025). *CNN vs Transformer for Wildfire Spread*. arXiv:2503.14150.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 1 — Environment Setup  (Camber: full compute resources)
# ══════════════════════════════════════════════════════════════
import gc, json, os, sys, time, warnings
from copy import deepcopy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torch.amp import autocast
from torch.cuda.amp import GradScaler
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', font_scale=1.1)
mpl.rcParams.update({'savefig.dpi': 300, 'figure.dpi': 150})
%matplotlib inline

# ── Project imports ──
sys.path.insert(0, os.path.abspath('..'))

from config import MODEL_CONFIG, TRAIN_CONFIG, SEED, MODELS_DIR, RESULTS_DIR, FIGURES_DIR
from src.data.dataset import get_dataloaders, FireSpreadDataset
from src.models.cellular_automata import CellularAutomataModel
from src.models.convlstm import ConvLSTMModel
from src.models.unet import UNetFire
from src.models.pi_cca import PIConvCellularAutomaton
from src.training.trainer import CombinedLoss, compute_metrics   # ★ FIX: was missing

# ── Load setup_config.json ──
_cfg_path = Path().resolve() / 'setup_config.json'
if not _cfg_path.exists():
    raise FileNotFoundError('setup_config.json not found — run 00_Setup.ipynb first')
cfg = json.load(open(_cfg_path))

PROCESSED_DIR    = Path(cfg['PROCESSED_DIR'])
FEATURE_CHANNELS = cfg['FEATURE_CHANNELS']
N_INPUT_CHANNELS = cfg['N_INPUT_CHANNELS']
CH               = cfg['CH']
norm_stats       = cfg['norm_stats']

# ── Device & random seeds ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)')
    torch.cuda.empty_cache()
else:
    vram_gb = 0
    print('CPU mode — no GPU detected')

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

print(f'Device: {device} | Seed: {SEED}')


## 3.1 Data Loading

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 2 — Data Loading (full dataset, larger batches on Camber)
# ══════════════════════════════════════════════════════════════

# Camber batch sizes (more VRAM available)
BASE_BS = 16 if vram_gb >= 8 else 8
BATCH_SIZES = {
    'convlstm': BASE_BS,
    'unet': BASE_BS,
    'pi_cca': max(8, BASE_BS // 2),  # still memory-hungry
    'ca': 32,
}

loaders = get_dataloaders(
    PROCESSED_DIR,
    batch_size=BASE_BS,
    num_workers=4,
    augment_train=True,
    seed=SEED,
    stats=norm_stats,
)

for name, loader in loaders.items():
    print(f'{name}: {len(loader.dataset)} samples, {len(loader)} batches (bs={loader.batch_size})')

x_batch, y_batch = next(iter(loaders['train']))
print(f'\nBatch: X={x_batch.shape}, Y={y_batch.shape}')
print(f'X range: [{x_batch.min():.3f}, {x_batch.max():.3f}]')
print(f'Fire pixel ratio (batch): {y_batch.mean():.4f}')

# Normalisation stats check
print('\n── Normalisation stats ──')
for ch_name in FEATURE_CHANNELS:
    s = norm_stats[ch_name]
    print(f'  {ch_name:>18s}: mean={s["mean"]:12.4f}, std={s["std"]:12.4f}')

## 3.2 Training Infrastructure

Enhanced trainer with:
- **AMP** (Automatic Mixed Precision)
- **Linear warmup + cosine annealing** LR schedule
- **Gradient accumulation** for larger effective batch sizes
- **Optimal threshold search** on validation set
- **Test-time augmentation** (TTA: 8-fold geometric transforms)
- **MC-Dropout uncertainty** for PI-CCA

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 3 — Enhanced Trainer (Camber: TTA + MC-Dropout + AMP)
# ══════════════════════════════════════════════════════════════
def compute_metrics_at_threshold(pred, target, threshold=0.5):
    with torch.no_grad():
        p_bin = (pred > threshold).float()
        t_bin = target.float()
        tp = (p_bin * t_bin).sum().item()
        fp = (p_bin * (1 - t_bin)).sum().item()
        fn = ((1 - p_bin) * t_bin).sum().item()
        tn = ((1 - p_bin) * (1 - t_bin)).sum().item()
        eps = 1e-8
        precision   = tp / (tp + fp + eps)
        recall      = tp / (tp + fn + eps)
        f1          = 2 * precision * recall / (precision + recall + eps)
        iou         = tp / (tp + fp + fn + eps)
        dice        = 2 * tp / (2 * tp + fp + fn + eps)
        accuracy    = (tp + tn) / (tp + fp + fn + tn + eps)
        specificity = tn / (tn + fp + eps)
    return {
        'iou': iou, 'dice': dice, 'f1': f1,
        'precision': precision, 'recall': recall,
        'accuracy': accuracy, 'specificity': specificity,
        'threshold': threshold,
    }

def find_optimal_threshold(preds, targets, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.05, 0.95, 0.025)
    best_dice, best_t = 0, 0.5
    for t in thresholds:
        m = compute_metrics_at_threshold(preds, targets, float(t))
        if m['dice'] > best_dice:
            best_dice = m['dice']
            best_t = float(t)
    return best_t, best_dice

def tta_predict(model, x, device, use_amp=True):
    """Test-time augmentation: 4 rotations × 2 flips = 8 predictions averaged."""
    preds = []
    for k in range(4):          # 0°, 90°, 180°, 270°
        x_rot = torch.rot90(x, k, [2, 3])
        for flip in [False, True]:
            x_aug = torch.flip(x_rot, [3]) if flip else x_rot
            x_aug = x_aug.to(device)
            if use_amp and torch.cuda.is_available():
                with autocast('cuda'):
                    pred = model(x_aug)
            else:
                pred = model(x_aug)
            pred = pred.float()
            if flip:
                pred = torch.flip(pred, [3])
            pred = torch.rot90(pred, -k, [2, 3])
            preds.append(pred.cpu())        # ★ FIX: move to CPU immediately
    return torch.stack(preds).mean(dim=0)


class CamberTrainer:
    """Full-resource trainer for Camber cloud: AMP, TTA, MC-Dropout."""

    def __init__(self, model, model_name, device, config, use_amp=True, accum_steps=1):
        self.model       = model.to(device)
        self.model_name  = model_name
        self.device      = device
        self.cfg         = config
        self.use_amp     = use_amp and torch.cuda.is_available()
        self.accum_steps = accum_steps
        self.criterion   = CombinedLoss(
            focal_w=config.get('focal_weight', 0.5),
            dice_w=config.get('dice_weight', 0.5),
            alpha=config.get('focal_alpha', 0.75),
            gamma=config.get('focal_gamma', 2.0),
        )
        params = [p for p in model.parameters() if p.requires_grad]
        self.has_params = len(params) > 0
        if self.has_params:
            self.optimizer = torch.optim.AdamW(
                params,
                lr=config.get('learning_rate', 3e-4),
                weight_decay=config.get('weight_decay', 1e-4),
            )
        else:
            self.optimizer = None
        self.scaler  = GradScaler() if self.use_amp else None   # ★ FIX: no 'cuda' arg
        self.history = {
            'train_loss': [], 'val_loss': [],
            'train_metrics': [], 'val_metrics': [],
            'lr': [], 'epoch_time': [],
        }

    def _get_lr(self, epoch, total_epochs, warmup_epochs=5):
        base_lr = self.cfg.get('learning_rate', 3e-4)
        min_lr  = 1e-6
        if epoch < warmup_epochs:
            return base_lr * (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return min_lr + 0.5 * (base_lr - min_lr) * (1 + np.cos(np.pi * progress))

    def train(self, train_loader, val_loader):
        epochs    = self.cfg.get('epochs', 80)
        patience  = self.cfg.get('early_stopping_patience', 15)
        warmup    = self.cfg.get('warmup_epochs', 5)
        grad_clip = self.cfg.get('gradient_clip', 1.0)
        best_val_dice, no_improve = 0.0, 0
        save_dir = MODELS_DIR / self.model_name
        save_dir.mkdir(parents=True, exist_ok=True)
        print(f'\n{"="*70}')
        print(f'Training {self.model_name} | {epochs} epochs | {self.device}')
        print(f'AMP={self.use_amp} | accum={self.accum_steps} | patience={patience}')
        print(f'{"="*70}')
        for epoch in range(epochs):
            t0 = time.time()
            lr = self._get_lr(epoch, epochs, warmup)
            if self.optimizer:
                for pg in self.optimizer.param_groups:
                    pg['lr'] = lr
            if self.has_params:
                train_loss, train_metrics = self._train_epoch(train_loader, grad_clip)
            else:
                train_loss, train_metrics = self._eval_epoch(train_loader)
            val_loss, val_metrics = self._eval_epoch(val_loader)
            dt = time.time() - t0
            self.history['train_loss'].append(float(train_loss))
            self.history['val_loss'].append(float(val_loss))
            self.history['train_metrics'].append(train_metrics)
            self.history['val_metrics'].append(val_metrics)
            self.history['lr'].append(float(lr))
            self.history['epoch_time'].append(float(dt))
            print(f'Epoch {epoch+1:3d}/{epochs} | '
                  f'Loss: {train_loss:.4f}/{val_loss:.4f} | '
                  f'Dice: {train_metrics["dice"]:.4f}/{val_metrics["dice"]:.4f} | '
                  f'IoU: {val_metrics["iou"]:.4f} | '
                  f'AUC(PR): {val_metrics.get("auc_pr", 0):.4f} | '
                  f'LR: {lr:.2e} | {dt:.1f}s')
            if val_metrics['dice'] > best_val_dice:
                best_val_dice = val_metrics['dice']
                no_improve = 0
                torch.save(self.model.state_dict(), save_dir / 'best_model.pt')
            else:
                no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1} (best Dice={best_val_dice:.4f})')
                break
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        with open(save_dir / 'training_history_camber.json', 'w') as f:
            json.dump(self.history, f, indent=2, default=str)
        print(f'Best val Dice: {best_val_dice:.4f}')
        return self.history

    def _train_epoch(self, loader, grad_clip=1.0):
        self.model.train()
        total_loss = 0
        all_preds, all_targets = [], []
        self.optimizer.zero_grad()
        for step, (x, y) in enumerate(tqdm(loader, desc='Train', leave=False)):
            x, y = x.to(self.device, non_blocking=True), y.to(self.device, non_blocking=True)
            if self.use_amp:
                with autocast('cuda'):
                    pred = self.model(x)
                    loss = self.criterion(pred, y) / self.accum_steps
                self.scaler.scale(loss).backward()
                if (step + 1) % self.accum_steps == 0:
                    self.scaler.unscale_(self.optimizer)
                    nn.utils.clip_grad_norm_(self.model.parameters(), grad_clip)
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                    self.optimizer.zero_grad()
            else:
                pred = self.model(x)
                loss = self.criterion(pred, y) / self.accum_steps
                loss.backward()
                if (step + 1) % self.accum_steps == 0:
                    nn.utils.clip_grad_norm_(self.model.parameters(), grad_clip)
                    self.optimizer.step()
                    self.optimizer.zero_grad()
            total_loss += loss.item() * self.accum_steps * x.size(0)
            # ★ FIX: move predictions to CPU immediately — prevents GPU OOM
            all_preds.append(pred.detach().float().cpu())
            all_targets.append(y.detach().float().cpu())
        avg_loss = total_loss / len(loader.dataset)
        metrics = compute_metrics(torch.cat(all_preds), torch.cat(all_targets))
        return avg_loss, metrics

    @torch.no_grad()
    def _eval_epoch(self, loader):
        self.model.eval()
        total_loss = 0
        all_preds, all_targets = [], []
        for x, y in tqdm(loader, desc='Eval', leave=False):
            x, y = x.to(self.device, non_blocking=True), y.to(self.device, non_blocking=True)
            if self.use_amp:
                with autocast('cuda'):
                    pred = self.model(x)
                    loss = self.criterion(pred.float(), y)
            else:
                pred = self.model(x)
                loss = self.criterion(pred, y)
            total_loss += loss.item() * x.size(0)
            # ★ FIX: move to CPU immediately
            all_preds.append(pred.float().cpu())
            all_targets.append(y.float().cpu())
        avg_loss = total_loss / len(loader.dataset)
        metrics = compute_metrics(torch.cat(all_preds), torch.cat(all_targets))
        return avg_loss, metrics

    @torch.no_grad()
    def evaluate_full(self, val_loader, test_loader, use_tta=True):
        """Full evaluation: threshold opt + optional TTA + test metrics."""
        self.model.eval()
        # ── Val predictions (no TTA for threshold finding) ──
        val_preds, val_targets = [], []
        for x, y in tqdm(val_loader, desc='Val eval', leave=False):
            x = x.to(self.device, non_blocking=True)
            if self.use_amp:
                with autocast('cuda'):
                    pred = self.model(x)
            else:
                pred = self.model(x)
            val_preds.append(pred.float().cpu())
            val_targets.append(y.float())
        val_preds   = torch.cat(val_preds)
        val_targets = torch.cat(val_targets)
        best_t, best_dice = find_optimal_threshold(val_preds, val_targets)
        val_metrics = compute_metrics_at_threshold(val_preds, val_targets, best_t)
        print(f'  Optimal threshold: {best_t:.3f} (val Dice={best_dice:.4f})')
        # ── Test predictions ──
        test_preds, test_targets = [], []
        desc = 'Test+TTA' if use_tta else 'Test'
        for x, y in tqdm(test_loader, desc=desc, leave=False):
            if use_tta:
                pred = tta_predict(self.model, x, self.device, self.use_amp)
            else:
                x = x.to(self.device, non_blocking=True)
                if self.use_amp:
                    with autocast('cuda'):
                        pred = self.model(x)
                else:
                    pred = self.model(x)
                pred = pred.float().cpu()
            test_preds.append(pred)
            test_targets.append(y.float())
        test_preds   = torch.cat(test_preds)
        test_targets = torch.cat(test_targets)
        test_metrics = compute_metrics_at_threshold(test_preds, test_targets, best_t)
        # Compare without TTA
        if use_tta:
            test_preds_noTTA, test_targets_noTTA = [], []
            for x, y in test_loader:
                x = x.to(self.device, non_blocking=True)
                if self.use_amp:
                    with autocast('cuda'):
                        pred = self.model(x)
                else:
                    pred = self.model(x)
                test_preds_noTTA.append(pred.float().cpu())
                test_targets_noTTA.append(y.float())
            test_no_tta = compute_metrics_at_threshold(
                torch.cat(test_preds_noTTA), torch.cat(test_targets_noTTA), best_t)
        else:
            test_no_tta = test_metrics
        return {
            'val': val_metrics,
            'test': test_metrics,
            'test_no_tta': test_no_tta,
            'optimal_threshold': best_t,
            'tta_used': use_tta,
        }


## 3.3 Model Registry & Hyperparameter Grid

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 4 — Models & HP grid (expanded for Camber resources)
# ══════════════════════════════════════════════════════════════

MODEL_CLASSES = {
    'ca': CellularAutomataModel,
    'convlstm': ConvLSTMModel,
    'unet': UNetFire,
    'pi_cca': PIConvCellularAutomaton,
}

print('Model parameter counts:')
for name, cls in MODEL_CLASSES.items():
    m = cls(config=MODEL_CONFIG[name])
    n_p = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'  {name:>10s}: {n_p:>10,d} params')
    del m

# ── Expanded HP grid for Camber ──
HP_GRID = {
    'convlstm': [
        {'learning_rate': 3e-4, 'weight_decay': 1e-4, 'focal_alpha': 0.75, 'focal_gamma': 2.0, 'dropout': 0.15},
        {'learning_rate': 1e-4, 'weight_decay': 5e-5, 'focal_alpha': 0.80, 'focal_gamma': 2.5, 'dropout': 0.10},
        {'learning_rate': 5e-4, 'weight_decay': 1e-4, 'focal_alpha': 0.70, 'focal_gamma': 2.0, 'dropout': 0.20},
        {'learning_rate': 2e-4, 'weight_decay': 1e-4, 'focal_alpha': 0.75, 'focal_gamma': 2.0, 'dropout': 0.10},
        {'learning_rate': 3e-4, 'weight_decay': 5e-5, 'focal_alpha': 0.80, 'focal_gamma': 2.0, 'dropout': 0.15},
    ],
    'unet': [
        {'learning_rate': 3e-4, 'weight_decay': 1e-4, 'focal_alpha': 0.75, 'focal_gamma': 2.0, 'dropout': 0.15},
        {'learning_rate': 1e-4, 'weight_decay': 5e-5, 'focal_alpha': 0.80, 'focal_gamma': 2.0, 'dropout': 0.10},
        {'learning_rate': 5e-4, 'weight_decay': 5e-5, 'focal_alpha': 0.75, 'focal_gamma': 2.5, 'dropout': 0.20},
        {'learning_rate': 2e-4, 'weight_decay': 1e-4, 'focal_alpha': 0.80, 'focal_gamma': 2.0, 'dropout': 0.15},
        {'learning_rate': 1e-4, 'weight_decay': 1e-4, 'focal_alpha': 0.75, 'focal_gamma': 2.5, 'dropout': 0.10},
    ],
    'pi_cca': [
        {'learning_rate': 3e-4, 'weight_decay': 1e-4, 'focal_alpha': 0.75, 'focal_gamma': 2.0, 'dropout': 0.15},
        {'learning_rate': 1e-4, 'weight_decay': 5e-5, 'focal_alpha': 0.80, 'focal_gamma': 2.0, 'dropout': 0.10},
        {'learning_rate': 2e-4, 'weight_decay': 1e-4, 'focal_alpha': 0.75, 'focal_gamma': 2.5, 'dropout': 0.15},
        {'learning_rate': 5e-4, 'weight_decay': 5e-5, 'focal_alpha': 0.75, 'focal_gamma': 2.0, 'dropout': 0.20},
        {'learning_rate': 1e-4, 'weight_decay': 1e-4, 'focal_alpha': 0.80, 'focal_gamma': 2.5, 'dropout': 0.10},
    ],
}

# Camber training settings
TUNING_EPOCHS = 12     # More thorough tuning phase
FINAL_EPOCHS  = 80     # Full training budget
PATIENCE      = 15     # More patient early stopping

candidate_models = ['convlstm', 'unet', 'pi_cca']

print(f'\nTraining plan: {candidate_models}')
print(f'Tuning: {TUNING_EPOCHS} epochs × {len(HP_GRID["unet"])} trials per model')
print(f'Final:  {FINAL_EPOCHS} epochs with early stopping (patience={PATIENCE})')
print(f'Batch sizes: {BATCH_SIZES}')

## 3.4 Training Loop — All Models

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 5 — Main training loop (Camber: full resources)
# ══════════════════════════════════════════════════════════════

all_results = {}
tuning_summary = {}

for model_name in candidate_models:
    print(f'\n{"═"*75}')
    print(f'  MODEL: {MODEL_CONFIG[model_name]["name"]}  ({model_name})')
    print(f'{"═"*75}')

    bs = BATCH_SIZES.get(model_name, BASE_BS)
    accum = 1  # no accumulation needed on Camber

    train_loader = DataLoader(
        loaders['train'].dataset, batch_size=bs, shuffle=True,
        num_workers=4, pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        loaders['val'].dataset, batch_size=bs, shuffle=False,
        num_workers=4, pin_memory=True,
    )
    test_loader = DataLoader(
        loaders['test'].dataset, batch_size=bs, shuffle=False,
        num_workers=4, pin_memory=True,
    )

    # ── Phase 1: Hyperparameter tuning ──
    trial_results = []
    for trial_idx, hp in enumerate(HP_GRID[model_name]):
        print(f'\n  Trial {trial_idx+1}/{len(HP_GRID[model_name])}: lr={hp["learning_rate"]}, '
              f'wd={hp["weight_decay"]}, alpha={hp["focal_alpha"]}, '
              f'gamma={hp["focal_gamma"]}, dropout={hp["dropout"]}')

        mcfg = deepcopy(MODEL_CONFIG[model_name])
        mcfg['dropout'] = hp.get('dropout', mcfg.get('dropout', 0.2))
        model = MODEL_CLASSES[model_name](config=mcfg).to(device)

        tcfg = {
            'epochs': TUNING_EPOCHS,
            'learning_rate': hp['learning_rate'],
            'weight_decay': hp['weight_decay'],
            'focal_alpha': hp['focal_alpha'],
            'focal_gamma': hp['focal_gamma'],
            'focal_weight': 0.5,
            'dice_weight': 0.5,
            'early_stopping_patience': TUNING_EPOCHS,
            'gradient_clip': 1.0,
            'warmup_epochs': 3,
        }

        trainer = CamberTrainer(
            model, model_name + '_tuning', device, tcfg,
            use_amp=True, accum_steps=accum,
        )
        hist = trainer.train(train_loader, val_loader)

        val_dices = [m['dice'] for m in hist['val_metrics']]
        best_dice = max(val_dices)
        trial_results.append({
            'hp': hp,
            'best_dice': float(best_dice),
            'last_dice': float(val_dices[-1]),
            'best_epoch': int(np.argmax(val_dices)) + 1,
        })
        print(f'    → Best val Dice: {best_dice:.4f} @ epoch {np.argmax(val_dices)+1}')

        del model, trainer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    best_trial = max(trial_results, key=lambda x: x['best_dice'])
    best_hp = best_trial['hp']
    tuning_summary[model_name] = {
        'selection_metric': 'dice',
        'best_trial': best_trial,
        'all_trials': trial_results,
    }
    print(f'\n  ★ Best config: {best_hp}')
    print(f'    Best tuning Dice: {best_trial["best_dice"]:.4f}')

    # ── Phase 2: Final training ──
    print(f'\n  Final training: {FINAL_EPOCHS} epochs...')
    mcfg = deepcopy(MODEL_CONFIG[model_name])
    mcfg['dropout'] = best_hp.get('dropout', mcfg.get('dropout', 0.2))
    model = MODEL_CLASSES[model_name](config=mcfg).to(device)

    tcfg = {
        'epochs': FINAL_EPOCHS,
        'learning_rate': best_hp['learning_rate'],
        'weight_decay': best_hp['weight_decay'],
        'focal_alpha': best_hp['focal_alpha'],
        'focal_gamma': best_hp['focal_gamma'],
        'focal_weight': 0.5,
        'dice_weight': 0.5,
        'early_stopping_patience': PATIENCE,
        'gradient_clip': 1.0,
        'warmup_epochs': 5,
    }

    trainer = CamberTrainer(
        model, model_name, device, tcfg,
        use_amp=True, accum_steps=accum,
    )
    history = trainer.train(train_loader, val_loader)

    # ── Phase 3: Full evaluation (threshold opt + TTA) ──
    ckpt_path = MODELS_DIR / model_name / 'best_model.pt'
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))

    use_tta = (model_name != 'ca')  # no TTA for CA baseline
    eval_results = trainer.evaluate_full(val_loader, test_loader, use_tta=use_tta)

    # TTA improvement
    if use_tta:
        tta_gain = eval_results['test']['dice'] - eval_results['test_no_tta']['dice']
        print(f'  TTA Dice gain: {tta_gain:+.4f}')

    all_results[model_name] = {
        'history': history,
        'tuning': tuning_summary[model_name],
        'eval': eval_results,
    }

    # Save
    save_dir = MODELS_DIR / model_name
    with open(save_dir / 'hyperparameter_tuning_camber.json', 'w') as f:
        json.dump(tuning_summary[model_name], f, indent=2, default=str)
    with open(save_dir / 'training_history_camber.json', 'w') as f:
        json.dump(history, f, indent=2, default=str)
    with open(save_dir / 'eval_results_camber.json', 'w') as f:
        json.dump(eval_results, f, indent=2, default=str)

    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── Physics baseline (CA) ──
print(f'\n{"═"*75}')
print('  MODEL: Cellular Automata (physics baseline)')
print(f'{"═"*75}')
ca_model = CellularAutomataModel(config=MODEL_CONFIG['ca']).to(device)
ca_trainer = CamberTrainer(ca_model, 'ca', device, TRAIN_CONFIG, use_amp=False)
ca_eval = ca_trainer.evaluate_full(loaders['val'], loaders['test'], use_tta=False)
all_results['ca'] = {'eval': ca_eval}

print('\n✅ All models trained and evaluated!')

## 3.5 MC-Dropout Uncertainty Estimation (PI-CCA)

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 6 — MC-Dropout uncertainty for PI-CCA
# ══════════════════════════════════════════════════════════════

ckpt = MODELS_DIR / 'pi_cca' / 'best_model.pt'
if ckpt.exists():
    pi_cca_model = PIConvCellularAutomaton(config=MODEL_CONFIG['pi_cca']).to(device)
    pi_cca_model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))

    # Get a test batch
    x_test, y_test = next(iter(loaders['test']))
    x_sample = x_test[:4].to(device)

    # MC-Dropout: 30 forward passes with dropout enabled
    n_mc = 30
    mean_pred, std_pred = pi_cca_model.predict_with_uncertainty(x_sample, n_samples=n_mc)

    mean_pred = mean_pred.cpu().numpy()
    std_pred = std_pred.cpu().numpy()
    gt = y_test[:4].numpy()

    # Visualise
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    titles = ['Input Fire', 'Mean Prediction', 'Uncertainty (σ)', 'Ground Truth']

    opt_t = all_results.get('pi_cca', {}).get('eval', {}).get('optimal_threshold', 0.5)

    for i in range(4):
        fire_in = x_test[i, CH['prev_fire_mask']].numpy()
        axes[i, 0].imshow(fire_in, cmap='hot', vmin=0, vmax=max(1, fire_in.max()))
        axes[i, 1].imshow(mean_pred[i, 0] > opt_t, cmap='hot', vmin=0, vmax=1)
        im = axes[i, 2].imshow(std_pred[i, 0], cmap='magma', vmin=0)
        plt.colorbar(im, ax=axes[i, 2], fraction=0.046)
        axes[i, 3].imshow(gt[i, 0], cmap='hot', vmin=0, vmax=1)

        for j in range(4):
            axes[i, j].axis('off')
            if i == 0:
                axes[i, j].set_title(titles[j], fontweight='bold')

    plt.suptitle(f'PI-CCA MC-Dropout Uncertainty ({n_mc} samples)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'pi_cca_uncertainty_camber.png', dpi=200, bbox_inches='tight')
    plt.show()

    del pi_cca_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print('PI-CCA checkpoint not found — skipping uncertainty analysis.')

## 3.6 Results Summary (Publication Table)

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 7 — Publication-ready results table
# ══════════════════════════════════════════════════════════════

rows = []
for name in ['ca', 'convlstm', 'unet', 'pi_cca']:
    if name not in all_results:
        continue
    r = all_results[name]
    test_m = r['eval'].get('test', {})
    test_no_tta = r['eval'].get('test_no_tta', test_m)
    val_m = r['eval'].get('val', {})
    opt_t = r['eval'].get('optimal_threshold', 0.5)

    rows.append({
        'Model': MODEL_CONFIG[name]['name'],
        'Type': MODEL_CONFIG[name].get('type', ''),
        'τ': f'{opt_t:.2f}',
        'Val Dice': f"{val_m.get('dice', 0):.4f}",
        'Test Dice': f"{test_no_tta.get('dice', 0):.4f}",
        'Test Dice+TTA': f"{test_m.get('dice', 0):.4f}",
        'Test IoU': f"{test_m.get('iou', 0):.4f}",
        'Precision': f"{test_m.get('precision', 0):.4f}",
        'Recall': f"{test_m.get('recall', 0):.4f}",
        'F1': f"{test_m.get('f1', 0):.4f}",
    })

results_df = pd.DataFrame(rows)
print('\n' + '='*90)
print('  TEST SET RESULTS — Camber Training (with TTA)')
print('='*90)
display(results_df)

# Save
results_df.to_csv(RESULTS_DIR / 'test_results_camber.csv', index=False)
with open(RESULTS_DIR / 'all_results_camber.json', 'w') as f:
    json.dump({k: v['eval'] for k, v in all_results.items()}, f, indent=2, default=str)

# LaTeX table for paper
print('\n── LaTeX table ──')
print(results_df.to_latex(index=False, float_format='%.4f'))

## 3.7 Training Curves

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 8 — Training curves (publication quality)
# ══════════════════════════════════════════════════════════════

colors = {'convlstm': '#2196F3', 'unet': '#4CAF50', 'pi_cca': '#FF5722'}
labels = {k: MODEL_CONFIG[k]['name'] for k in colors}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for name in candidate_models:
    if name not in all_results or 'history' not in all_results[name]:
        continue
    h = all_results[name]['history']
    c = colors[name]
    lbl = labels[name]
    epochs_range = range(1, len(h['train_loss']) + 1)

    # Loss
    axes[0, 0].plot(epochs_range, h['train_loss'], color=c, alpha=0.6, label=f'{lbl} (train)')
    axes[0, 0].plot(epochs_range, h['val_loss'], color=c, linestyle='--', label=f'{lbl} (val)')

    # Dice
    val_dice = [m['dice'] for m in h['val_metrics']]
    train_dice = [m['dice'] for m in h['train_metrics']]
    axes[0, 1].plot(epochs_range, train_dice, color=c, alpha=0.6, label=f'{lbl} (train)')
    axes[0, 1].plot(epochs_range, val_dice, color=c, linestyle='--', label=f'{lbl} (val)')

    # IoU
    val_iou = [m['iou'] for m in h['val_metrics']]
    axes[1, 0].plot(epochs_range, val_iou, color=c, label=lbl)

    # LR
    axes[1, 1].plot(epochs_range, h['lr'], color=c, label=lbl)

axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training & Validation Loss', fontweight='bold')
axes[0, 0].legend(fontsize=7)

axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Dice')
axes[0, 1].set_title('Dice Score', fontweight='bold')
axes[0, 1].legend(fontsize=7)

axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('IoU')
axes[1, 0].set_title('Validation IoU', fontweight='bold')
axes[1, 0].legend(fontsize=8)

axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('LR')
axes[1, 1].set_title('Learning Rate (Warmup + Cosine)', fontweight='bold')
axes[1, 1].legend(fontsize=8)
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'training_curves_camber.png', dpi=300, bbox_inches='tight')
plt.show()

## 3.8 Qualitative Predictions + TTA Comparison

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 9 — Visual predictions (with & without TTA)
# ══════════════════════════════════════════════════════════════

x_test, y_test = next(iter(loaders['test']))
n_samples = min(4, x_test.shape[0])
n_models = len(candidate_models)

fig, axes = plt.subplots(n_samples, n_models + 2, figsize=(4*(n_models+2), 4*n_samples))
if n_samples == 1:
    axes = axes[np.newaxis, :]

for row in range(n_samples):
    fire_in = x_test[row, CH['prev_fire_mask']].numpy()
    gt = y_test[row].squeeze().numpy()

    axes[row, 0].imshow(fire_in, cmap='hot', vmin=0, vmax=max(1, fire_in.max()))
    if row == 0: axes[row, 0].set_title('Input Fire', fontweight='bold')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(gt, cmap='hot', vmin=0, vmax=1)
    if row == 0: axes[row, 1].set_title('Ground Truth', fontweight='bold')
    axes[row, 1].axis('off')

    for col, mname in enumerate(candidate_models):
        ckpt = MODELS_DIR / mname / 'best_model.pt'
        if not ckpt.exists():
            continue

        model = MODEL_CLASSES[mname](config=MODEL_CONFIG[mname]).to(device)
        model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
        model.eval()

        opt_t = all_results.get(mname, {}).get('eval', {}).get('optimal_threshold', 0.5)

        # Use TTA for final predictions
        with torch.no_grad():
            pred = tta_predict(model, x_test[row:row+1], device, use_amp=True)
        pred_np = pred.squeeze().cpu().numpy()

        axes[row, col+2].imshow(pred_np > opt_t, cmap='hot', vmin=0, vmax=1)
        if row == 0:
            axes[row, col+2].set_title(f'{MODEL_CONFIG[mname]["name"]}\n(τ={opt_t:.2f}, +TTA)', fontweight='bold', fontsize=9)
        axes[row, col+2].axis('off')
        del model

plt.suptitle('Test Set Predictions — Camber (with TTA)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'predictions_camber.png', dpi=300, bbox_inches='tight')
plt.show()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 3.9 Threshold Sensitivity Analysis

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 10 — Threshold sensitivity
# ══════════════════════════════════════════════════════════════

thresholds = np.arange(0.05, 0.95, 0.025)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for mname in candidate_models:
    ckpt = MODELS_DIR / mname / 'best_model.pt'
    if not ckpt.exists():
        continue

    model = MODEL_CLASSES[mname](config=MODEL_CONFIG[mname]).to(device)
    model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    model.eval()

    val_preds, val_targets = [], []
    with torch.no_grad():
        for x, y in loaders['val']:
            x = x.to(device)
            if torch.cuda.is_available():
                with autocast('cuda'):
                    pred = model(x)
            else:
                pred = model(x)
            val_preds.append(pred.float().cpu())
            val_targets.append(y)

    val_preds = torch.cat(val_preds)
    val_targets = torch.cat(val_targets)

    dices, ious, f1s = [], [], []
    for t in thresholds:
        m = compute_metrics_at_threshold(val_preds, val_targets, float(t))
        dices.append(m['dice'])
        ious.append(m['iou'])
        f1s.append(m['f1'])

    c = colors[mname]
    axes[0].plot(thresholds, dices, color=c, marker='.', markersize=3, label=labels[mname])
    axes[1].plot(thresholds, ious, color=c, marker='.', markersize=3, label=labels[mname])
    axes[2].plot(thresholds, f1s, color=c, marker='.', markersize=3, label=labels[mname])

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

for ax, metric in zip(axes, ['Dice', 'IoU', 'F1']):
    ax.set_xlabel('Threshold τ')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} vs Threshold (Validation)', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'threshold_sensitivity_camber.png', dpi=300, bbox_inches='tight')
plt.show()

## 3.10 Tuning Summary — All Models

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 11 — Tuning summary (HP search results)
# ══════════════════════════════════════════════════════════════

print('\n' + '='*80)
print('  HYPERPARAMETER TUNING SUMMARY')
print('='*80)

for model_name in candidate_models:
    if model_name not in tuning_summary:
        continue
    ts = tuning_summary[model_name]
    print(f'\n{MODEL_CONFIG[model_name]["name"]}:')
    print(f'  Best trial Dice: {ts["best_trial"]["best_dice"]:.4f}')
    print(f'  Best HP: {ts["best_trial"]["hp"]}')
    print(f'  All trials:')
    for i, trial in enumerate(ts['all_trials']):
        marker = ' ★' if trial == ts['best_trial'] else ''
        print(f'    [{i+1}] Dice={trial["best_dice"]:.4f} @ epoch {trial["best_epoch"]} | '
              f'lr={trial["hp"]["learning_rate"]}, wd={trial["hp"]["weight_decay"]}{marker}')

# Save global summary
with open(RESULTS_DIR / 'tuning_summary_camber.json', 'w') as f:
    json.dump(tuning_summary, f, indent=2, default=str)

print(f'\nSaved to {RESULTS_DIR / "tuning_summary_camber.json"}')

## Summary

### Key improvements over previous versions

1. **Full dataset** — all 14,979 training samples used (no subsampling)
2. **Mixed precision (AMP)** — faster training, lower memory usage
3. **All 4 models** trained including PI-CCA (the novel contribution)
4. **Linear warmup (5 epochs) + cosine decay** LR schedule
5. **5 structured HP trials** per model with 12-epoch tuning phase
6. **80-epoch budget** with patient early stopping (15 epochs)
7. **Optimal threshold τ** per model via validation sweep
8. **Test-time augmentation** (8-fold TTA) for all DL models
9. **MC-Dropout uncertainty** estimation for PI-CCA
10. **Publication-ready** figures (DPI=300) and LaTeX result tables

### Expected improvements vs FastTrack

| Metric | FastTrack (10 ep, 5K samples) | Camber (80 ep, full data) |
|--------|------------------------------|---------------------------|
| U-Net Dice | ~0.32 | 0.38–0.42 (expected) |
| ConvLSTM Dice | ~0.28 | 0.33–0.37 (expected) |
| PI-CCA Dice | skipped | 0.35–0.40 (expected) |

### Next steps

- **Notebook 04**: Detailed test-set evaluation & statistical tests
- **Notebook 05**: SHAP/Grad-CAM explainability analysis